# 🚀 Different work, or the same work with AI?
### Assignment after session 2 · Module 2

Since ChatGPT came out in November 2022, many more new startups say that AI is what they sell. A chart of how often people
say "AI" cannot tell you what that means:

> **Did startups start doing different work, or do they do the same work as before, now with AI?**

If new work explains the rise, founders moved to new markets. If the same work explains it, AI is spreading through existing
markets. Economists call that a general-purpose technology: one that many industries adopt for their own work, as they did with electricity.

**The data.** Y Combinator (YC) is a well-known US startup programme. It takes a group of startups several times a year, and each
group is called a **batch**. We use every YC startup since 2005, each described in a few plain sentences. No single tool can answer the question: a language model is too slow for everything, a topic model does not know what AI is, and a classifier needs labels. So you
combine them, and each tool does the job it is good at:

| Step | Tool | Job | Runs for |
|---|---|---|---|
| 1 | Decision model | Is AI the core? For every startup | about 7 min |
| 2 | Small LLM | Rewrite a sample as plain work | about 5 min |
| 3 | BERTopic, with the LLM naming topics | Find the types of work | about 2 min |
| 4 | Decision model, classifier | Label every startup with a type | about 12 min |
| 5 | Shift-share | Answer the question | seconds |

**Hand in by Sunday 8 November, 23:59, on Moodle:** one PDF of one page (step 6), with your names and a link to this notebook,
showing its outputs. Work in groups of up to three. You must hand it in to take the exam. It gets no grade.

You may use AI to help with code. The cells marked ✏️ are yours. Fill them in yourselves, because they are the point of the exercise: the
checks you do by hand, the definitions, and your conclusion.

**Before you start:** Runtime → Change runtime type → T4 GPU. Then run the cells one at a time.
If Colab asks for access to `HF_TOKEN`, click *Cancel*.

**Own machine.** This notebook also runs in Jupyter or VS Code on a laptop with Ollama installed: Moodle explains how. Leave `BACKEND` on `ollama`.

**Plan B, if local does not work** (no free GPU, or Ollama fails). Create a free key at [aistudio.google.com](https://aistudio.google.com) and
store it as a Colab secret called `GEMINI_API_KEY` (key icon on the left, then switch on access for this notebook). In the setup cell, set `BACKEND`
to `gemini` and run everything again. A Groq key from [console.groq.com](https://console.groq.com), stored as `GROQ_API_KEY`, works the same way with `groq`.

The code stays the same and your texts go to the provider. Free keys allow few calls a minute, so plan B runs on smaller samples and takes longer. Your numbers will differ from the slides, because the cloud model is stronger.

The cloud has no decision model, so here plan B replays saved answers from a real `d3:2b` run. Those parts run instantly, and new questions to the decision model do not work.

In [ ]:
# @title Setup: install Ollama and download the models (2 to 4 minutes) { display-mode: "form" }
BACKEND = "ollama"  # @param ["ollama", "gemini", "groq"]
%pip install -q -U openai bertopic
import hashlib, json, os, random, shutil, subprocess, sys, time, warnings
from concurrent.futures import ThreadPoolExecutor
from typing import Literal, Optional
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.model_selection import cross_val_score
from sklearn.svm import LinearSVC
warnings.filterwarnings("ignore")

OLLAMA = "http://localhost:11434"
DECIDER = "d3:2b"
DEVICE = "cpu" if sys.platform == "darwin" else None   # on a Mac, Ollama already fills the GPU memory
if BACKEND == "ollama":
    if shutil.which("ollama") is None and "google.colab" in sys.modules:
        subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
    if shutil.which("ollama") is None:
        raise RuntimeError("Ollama is not installed. On your own machine, install it as described on Moodle. Otherwise use plan B: see the top of this notebook.")

    # start_new_session keeps the server alive when you stop a cell
    subprocess.Popen(["ollama", "serve"], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    for model in ["d3:2b", "gemma4:e2b"]:
        t0 = time.time()
        done = subprocess.run(["ollama", "pull", model], capture_output=True)
        print(f"{model} ready ({time.time() - t0:.0f} s)" if done.returncode == 0 else f"{model} did not download. Use plan B: see the top of this notebook.")

    if "google.colab" in sys.modules:
        gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
        print(gpu if gpu else "No GPU on this runtime. Each text would take 10 to 20 seconds, which is too slow here. Use Runtime → Change runtime type → T4 GPU. If Colab has none free, use plan B: see the top of this notebook.")
    BASE_URL, API_KEY, MODEL, EXTRA, PAUSE = OLLAMA + "/v1", "ollama", "gemma4:e2b", {"reasoning_effort": "none"}, 0
else:   # plan B: the same calls go to a cloud model. Free keys allow 15 (Gemini) or 30 (Groq) calls a minute
    BASE_URL, secret, MODEL, EXTRA, PAUSE = {
        "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai", "GEMINI_API_KEY", "gemini-flash-lite-latest", {}, 4.2),
        "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "openai/gpt-oss-20b", {"reasoning_effort": "low"}, 3),
    }[BACKEND]
    try:
        from google.colab import userdata
        API_KEY = userdata.get(secret)
    except ImportError:
        API_KEY = os.environ[secret]

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6EC", "legend.frameon": False})
pd.set_option("display.max_colwidth", 200)

The `ask()` and `decide()` functions from the session notebooks.

In [ ]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(BASE_URL, "·", MODEL)



def ask(system, user, schema):
    for temperature in (0, 0.2, 0.4):   # the same temperature would repeat the same mistake
        time.sleep(PAUSE)
        try:
            response = client.chat.completions.parse(
                model=MODEL,
                temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                response_format=schema,
                **EXTRA,
            )
            return response.choices[0].message.parsed
        except Exception as e:
            error = e
            if "429" in str(e):
                time.sleep(30)   # over the free limit: wait for the next minute
    raise error



def decide(state, questions):
    payload = {"model": DECIDER, "state": state, "questions": questions}
    response = requests.post(OLLAMA + "/v1/systemone", json=payload, timeout=180)
    return response.json()["answers"]

---
# 📦 The data

Every company from the public YC directory. We keep those with a real description and a known batch. We group the batches into
periods: 2017–19, 2020–22, and 2023–26. ChatGPT came out at the end of 2022, so 2023 is the first batch year after it. An earlier period, 2011–16, is
in the data but not in the comparison.

In [ ]:
yc = pd.read_json("https://yc-oss.github.io/api/companies/all.json")
yc["text"] = yc["one_liner"].fillna("") + ". " + yc["long_description"].fillna("")
yc["year"] = yc["batch"].str.extract(r"(20[0-9][0-9])")[0].astype(float)
yc = yc[(yc["text"].str.split().str.len() >= 20) & (yc["industry"] != "Unspecified") & (yc["year"] <= 2026)].dropna(subset=["year"])
yc = yc.drop_duplicates("name").reset_index(drop=True)
yc["period"] = pd.cut(yc["year"], [2010, 2016, 2019, 2022, 2026], labels=["2011-16", "2017-19", "2020-22", "2023-26"])
print(len(yc), "startups")
yc.groupby("period", observed=True).size()

In [ ]:
yc[["name", "batch", "one_liner", "long_description"]].sample(4, random_state=1)

---
# ⚡ Step 1 · Decide: is AI the core?

A yes-or-no question for every startup. This is a job for a decision model: one fast answer, no text needed. We count AI as the core
when the product is built on AI. A side feature does not count.

The first call loads the model, which can take up to three minutes. After that, the code sends four questions at a time.

In [ ]:
AI_QUESTION = "Is artificial intelligence the core of what this startup sells?"

def is_ai(text):
    return decide(text, {"ai": {"type": "noul", "instructions": AI_QUESTION}})["ai"]["noul"]

t0 = time.time()
if BACKEND == "ollama":
    is_ai("warming up the model")
    with ThreadPoolExecutor(4) as pool:
        yc["ai_prob"] = list(pool.map(is_ai, yc["text"]))
else:   # plan B: the cloud has no decision model, so we load the answers saved from a real d3:2b run
    saved = pd.read_csv("https://raw.githubusercontent.com/aaubs/ds-master/main/data/llm_session/ai_core_yc.csv").set_index("name")["ai_prob"]
    yc["ai_prob"] = yc["name"].map(saved)
    yc = yc.dropna(subset=["ai_prob"]).reset_index(drop=True)
yc["ai"] = yc["ai_prob"] > 0.5
yc[["name", "ai_prob"]].to_csv("step1_ai.csv", index=False)   # save as you go
print(f"{(time.time() - t0) / 60:.1f} minutes")

In [ ]:
# @title Chart: AI as the core, by batch year { display-mode: "form" }
share = yc[yc["year"] >= 2014].groupby("year")["ai"].mean() * 100
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(share.index, share.values, "-o", color=NAVY)
ax.axvline(2022.9, color=ORANGE, ls="--"); ax.text(2022.7, 8, "ChatGPT", color=ORANGE, ha="right")
ax.set_ylabel("% of the batch"); ax.set_ylim(0, 100)
plt.show()

✏️ **Check by hand.** The chart is only as good as the labels. The next cell prints 30 random startups, without the model's answer.
Read them and decide yourselves: is AI the core? Then write `True` or `False` for each one, in the order shown, in the cell after it.

In [ ]:
check_ai = yc[yc["year"] >= 2017].sample(30, random_state=8)[["name", "year", "text"]].reset_index(drop=True)
for r in check_ai.itertuples():
    print(f"{r.Index:2} {r.name} ({r.year:.0f}): {r.text[:280]}\n")

In [ ]:
MY_AI = [ ]   # ✏️ 30 times True or False, in the order above

In [ ]:
model_ai = yc.set_index("name").loc[check_ai["name"], "ai"].tolist()
if len(MY_AI) == len(model_ai):
    print(f"agreement {accuracy_score(MY_AI, model_ai):.0%} · kappa {cohen_kappa_score(MY_AI, model_ai):.2f}")
    print(f"you say yes {np.mean(MY_AI):.0%} · the model says yes {np.mean(model_ai):.0%}")
else:
    print("Fill in MY_AI first.")

---
# ✍️ Step 2 · Extract: what work does it really do?

Startup descriptions are written to impress. *AI-native platform for the future of work* tells a topic model nothing. So we ask the
small LLM to rewrite each description as plain work: a verb and an object.

This is the slow step, so it gets a **sample**: 200 startups per period. That is the rule for LLMs: spend them on a sample where you need
text, and let cheaper models do the volume.

In [ ]:
class Work(BaseModel):
    task: str = Field(description="The work the product does for its customer: a verb and an object, at most 8 plain words.")

WORK_INSTRUCTION = '''You read a short startup description and say what work the product does for its customer.
Write a verb and an object, at most 8 plain words. Say the concrete work, not the technology and not the marketing.
Do not use the words AI, platform, solution or powered.
Examples: answer customer support tickets; review legal contracts; deliver groceries to homes; sell term life insurance; test new cancer drugs.'''

sample = yc[yc["period"] != "2011-16"].groupby("period", observed=True).sample(200 if BACKEND == "ollama" else 100, random_state=7).reset_index(drop=True)
print(sample["text"].iloc[:5].map(lambda t: ask(WORK_INSTRUCTION, t, Work).task).tolist())

✏️ *The cell above printed five answers. If they are too vague, change `WORK_INSTRUCTION` and run it again, before you spend five minutes on all of them.*

In [ ]:
t0 = time.time()
with ThreadPoolExecutor(2 if BACKEND == "ollama" else 1) as pool:
    sample["task"] = list(pool.map(lambda t: ask(WORK_INSTRUCTION, t, Work).task, sample["text"]))
sample[["name", "task"]].to_csv("step2_tasks.csv", index=False)
print(f"{(time.time() - t0) / 60:.1f} minutes")
sample[["name", "one_liner", "task"]].sample(10, random_state=2)

---
# 🗺️ Step 3 · Discover: which types of work exist?

We do not want to invent the categories, so we let **BERTopic** find groups. New today: the LLM names each topic. BERTopic sends it the
keywords and a few typical documents for each topic, so naming takes one call per topic and not one per startup.

We compare the raw descriptions with the plain work. Then you judge the groups: the model groups, and you decide what the groups mean.

In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI as LLMNames
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

def topic_model(seed=42, names=False):
    aspects = {"Main": KeyBERTInspired(), "MMR": MaximalMarginalRelevance(diversity=0.4)}
    if names:
        aspects["Label"] = LLMNames(client, model=MODEL, nr_docs=8, prompt='''These startups do similar work:
[DOCUMENTS]
Keywords: [KEYWORDS]
Name the type of work in two to five plain words. Do not use the words AI or platform. Reply with the name only.''',
                                    delay_in_seconds=PAUSE, generator_kwargs={"temperature": 0, **EXTRA})
    return BERTopic(embedding_model=embedder, representation_model=aspects,
                    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0, metric="cosine", random_state=seed),
                    hdbscan_model=HDBSCAN(min_cluster_size=10, cluster_selection_method="leaf", prediction_data=True),
                    vectorizer_model=CountVectorizer(stop_words="english", ngram_range=(1, 2)))

for name, column in [("raw descriptions", "text"), ("plain work", "task")]:
    topics, _ = topic_model().fit_transform(sample[column].tolist())
    topics = np.array(topics)
    print(f"{name:<18} {topics.max() + 1} topics, {(topics == -1).mean():.0%} without a topic")

In [ ]:
docs = sample["task"].tolist()
embeddings = embedder.encode(docs, show_progress_bar=False)
model = topic_model(names=True)
topics, _ = model.fit_transform(docs, embeddings)
sample["topic"] = model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0.45)
print(f"without a topic: {np.mean(np.array(topics) == -1):.0%} before, {(sample['topic'] == -1).mean():.0%} after reduce_outliers")

label = lambda t: model.topic_aspects_["Label"][t][0][0] if t >= 0 else "(no topic)"
table = pd.crosstab(sample["topic"], sample["period"], normalize="columns").round(3)
table.insert(0, "name", [label(t) for t in table.index])
table.insert(1, "n", sample["topic"].value_counts().reindex(table.index))
table.insert(2, "examples", [" · ".join(sample.loc[sample["topic"] == t, "task"].head(3)) for t in table.index])
table

✏️ **Name your types of work.** Merge the topics into **8 to 10 types** that make sense to you. Read the names and the examples, and do not
trust a name blindly: the LLM may call a group of security tools "property monitoring". In `MERGE` below, write each topic number with
your type name. Several numbers can share one type. Leave out topics you cannot interpret: those startups get no type here, and the
models in step 4 will label them.

`DEFINITIONS` gives each type one line, which the decision model will use in step 4. Do not add an *Other* option: when we tested this, the model chose it far too often. Without it, the model matched
labels made by careful reviewers on 79 % of startups, up from 67 %.

In [ ]:
MERGE = {
    # topic number: "type of work", for example  3: "Food and consumer",
}

DEFINITIONS = {
    # "type of work": "one line that says what belongs here",
}

In [ ]:
sample["type"] = sample["topic"].map(MERGE)
print(sample["type"].value_counts(dropna=False))

---
# 🏷️ Step 4 · Scale: label every startup

So far only the sample has types. Two ways to reach all startups, and you pick the better one.

- **Classifier:** the sample now has labels, so train the classifier you know from session 1 on them. It costs seconds.
- **Decision model:** it needs no training, only your definitions, and it takes about 12 minutes.

Then you check both by hand, and you use the one that agrees with you more often.

In [ ]:
labelled = sample.dropna(subset=["type"])
everything = embedder.encode(yc["text"].tolist(), batch_size=128, show_progress_bar=False)
position = dict(zip(yc["name"], range(len(yc))))
X = everything[[position[n] for n in labelled["name"]]]
classifier = LinearSVC()
print(f"cross-validated on the sample: {cross_val_score(classifier, X, labelled['type'], cv=5).mean():.0%}")
classifier.fit(X, labelled["type"])
yc["type_classifier"] = classifier.predict(everything)

In [ ]:
def work_type(text):
    a = decide(text, {"t": {"type": "choice", "instructions": "What type of work does this product mainly do for its customer?",
                            "criteria": DEFINITIONS}})["t"]
    return a["choice"], a["confidence"]

if BACKEND == "ollama":
    work_type("warming up the model")
    t0 = time.time()
    with ThreadPoolExecutor(4) as pool:
        answers = list(pool.map(work_type, yc["text"]))
    yc["type_decision"] = [a[0] for a in answers]
    yc["confidence"] = [a[1] for a in answers]
    yc[["name", "type_classifier", "type_decision", "confidence"]].to_csv("step4_types.csv", index=False)
    print(f"{(time.time() - t0) / 60:.1f} minutes · the two agree on {(yc['type_classifier'] == yc['type_decision']).mean():.0%} of startups")
    print(f"confidence above 0.8: {(yc['confidence'] > 0.8).mean():.0%} of startups")
else:   # plan B: your definitions are new questions, so there are no saved answers. The classifier does this step alone
    yc["type_decision"] = yc["type_classifier"]
    print("Plan B: the decision model is not available here, so both columns hold the classifier's labels.")

✏️ **Check by hand.** The next cell prints 40 startups, without any model's answer, and your type names. Label each one with a type name, in the
order shown, in the cell after it.

In [ ]:
check_type = yc[yc["period"] != "2011-16"].sample(40, random_state=9)[["name", "text"]].reset_index(drop=True)
for r in check_type.itertuples():
    print(f"{r.Index:2} {r.name}: {r.text[:240]}\n")
print(list(DEFINITIONS))

In [ ]:
MY_TYPES = [ ]   # ✏️ 40 type names, in the order above

In [ ]:
if len(MY_TYPES) == len(check_type):
    look = yc.set_index("name").loc[check_type["name"]]
    for c in ["type_classifier", "type_decision"]:
        print(f"{c:<16} agrees with you on {accuracy_score(MY_TYPES, look[c]):.0%} · kappa {cohen_kappa_score(MY_TYPES, look[c]):.2f}")
else:
    print("Fill in MY_TYPES first.")

In [ ]:
yc["type"] = yc["type_classifier"]   # ✏️ pick the one that did better in your hand check

---
# 🧮 Step 5 · Answer the question

First the two pictures. Which types of work grew and shrank? And how far did AI spread inside each type?

In [ ]:
d = yc[yc["period"].isin(["2017-19", "2023-26"])]
before, after = d[d["period"] == "2017-19"], d[d["period"] == "2023-26"]
types = sorted(d["type"].unique())
compare = pd.DataFrame({
    "share before": before["type"].value_counts(normalize=True).reindex(types).fillna(0),
    "share after": after["type"].value_counts(normalize=True).reindex(types).fillna(0),
    "AI before": before.groupby("type")["ai"].mean().reindex(types).fillna(0),
    "AI after": after.groupby("type")["ai"].mean().reindex(types).fillna(0)})
compare.sort_values("share after", ascending=False).style.format("{:.0%}")

**Shift-share** is a standard tool from economics for exactly this question. We compare 2017–19 with 2023–26. The share of startups with AI
as the core rose, and we split that rise into two parts:

- **Within:** a type of work stays the same, but more of it uses AI. For each type: `average share of startups in this type × change in its AI share`
- **Between:** the mix moved towards types that already used more AI. For each type: `average AI share of this type × change in its share of startups`

"Average" means the average of the two periods. Add up each part over all types. The two parts add up to the total rise.

**A small example with two types.** Health has 80 % of startups before and 40 % after, with AI at 10 % and 20 %. Agents has 20 % before and 60 % after, with AI at
80 % and 90 %. Total AI share: 24 % before (0.8 × 10 + 0.2 × 80) and 62 % after (0.4 × 20 + 0.6 × 90), so a rise of 38 points.

- Within: health 0.6 × 10 = 6, plus agents 0.4 × 10 = 4, so **10 points**
- Between: health 15 × (−0.4) = −6, plus agents 85 × 0.4 = 34, so **28 points**

10 + 28 = 38. In this made-up case only 26 % of the rise happened inside the same types of work, so most of it is a move to different work.

In [ ]:
def shift_share(table):
    within = ((table["share before"] + table["share after"]) / 2 * (table["AI after"] - table["AI before"])).sum()
    between = ((table["AI before"] + table["AI after"]) / 2 * (table["share after"] - table["share before"])).sum()
    return within, between

within, between = shift_share(compare)
print(f"AI share: {before['ai'].mean():.0%} → {after['ai'].mean():.0%}")
print(f"within types {within * 100:+.0f} points · between types {between * 100:+.0f} points")
print(f"so {within / (within + between):.0%} of the rise happened inside the same type of work")

runs = []
for seed in range(500):   # resample the startups to see how much this could move by chance
    b_ = before.sample(frac=1, replace=True, random_state=seed)
    a_ = after.sample(frac=1, replace=True, random_state=seed + 1000)
    t = pd.DataFrame({"share before": b_["type"].value_counts(normalize=True).reindex(types).fillna(0),
                      "share after": a_["type"].value_counts(normalize=True).reindex(types).fillna(0),
                      "AI before": b_.groupby("type")["ai"].mean().reindex(types).fillna(0),
                      "AI after": a_.groupby("type")["ai"].mean().reindex(types).fillna(0)})
    w, btw = shift_share(t)
    runs.append(w / (w + btw))
print(f"95 % of resamples between {np.percentile(runs, 2.5):.0%} and {np.percentile(runs, 97.5):.0%}")

In [ ]:
# @title Chart: where did AI spread? { display-mode: "form" }
c = compare.sort_values("AI after")
fig, ax = plt.subplots(figsize=(7, 4))
ax.hlines(c.index, c["AI before"] * 100, c["AI after"] * 100, color=LAV, lw=3)
ax.scatter(c["AI before"] * 100, c.index, color=GREY, label="2017-19", zorder=3)
ax.scatter(c["AI after"] * 100, c.index, color=NAVY, label="2023-26", zorder=3)
ax.set_xlabel("% of startups with AI at the core"); ax.legend(); ax.grid(axis="y", visible=False)
plt.show()

---
# 📝 Step 6 · One page

Write one page, with these four parts:

1. **Answer.** Different work, or the same work with AI? Give your numbers and your best chart.
2. **Which tool did what, and why.** For each step: why this tool and not another? How long did it take?
3. **What you found by hand.** What did your two checks by hand show? Name one case where a model was wrong and very confident.
4. **What you do not trust.** Give at least these three problems. Does a startup selling "AI agents for accountants" belong to AI or to finance, and how
   would your answer change with the other choice? Then, the descriptions are today's text, and a company from 2018 may have
   rewritten its description since then. And a startup may say AI is its core because that helps it raise money.

To hand in: in Colab, choose File → Save, then Share → *Anyone with the link can view*. Put the link and your names on the page, save it as a PDF,
and upload it on Moodle.